In [1]:
%pip install torch torchvision torchaudio
%pip install scikit-learn
%pip install torch.utils
%pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# PyTorch Regression with Sklearn California Housing
## Goal

Build a regression model using `PyTorch` to predict a target variable from the California Housing Dataset. 
This will integrate `pandas` for data handling and `scikit-learn` for preprocessing before feeding the data into a PyTorch neural network.

## Tasks

1. **Data Fetching:** Fetch the diabetes dataset from sklearn and convert it to a `pandas DataFrame`.

2. **Data Preprocessing:** 

    * Separate features (X) and target (y).
    * Split the dataset into training and testing sets using train_test_split.
    * Scale features (X) using StandardScaler.

3. **PyTorch Data Preparation:**

    * Convert preprocessed NumPy Arrays into Tensor objects.
    * Create custom Datatset class to handle data.
    * Create DataLoader for training and testing datasets.

4. **Define Neural Network:** Complete `__init__` and `forward` methods of the `HousingRegressionModel` class. 
    Use `nn.Linear` layers, and adjust input features for the preprocessed data.

5. **Define Loss Function and Optimizer:** Choose and create appropriate objects for regression.

6. **Training Loop:** Write the training loop for ### epochs, iterating through the `DataLoader`.

7. **Evaluate and Vizualize:** Assess model performance on the test set and visualize training loss and predictions


# 1. Import Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2. Data Loading
Load the `california_housing` dataset from `sklearn.datasets` and convert it to a `pandas.DataFrame`. 
Display general information

In [3]:
# Fetches california housing directly as a Pandas DataFrame
housing_df = fetch_california_housing(as_frame = True).frame

In [4]:
# Display first 5 rows of housing data
print(housing_df.head())
print()

# Display general information of housing data
print(housing_df.info())
print()

# Display statistical information of housing data
print(housing_df.describe())
print()

# Display size of the housing data
print(housing_df.size)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 n

## 3. Data Preprocessing with Pandas and Scikit-learn
**Do:**
1. Separate the features (X) and the target (y) from the DF. 
2. Split the data into training and testing sets using tts. 
3. Apply `StandardScaler` to the training and testing features. This improves neural network performance

In [ ]:
# Set features DF and target series
X = housing_df[housing_df.columns[:-1]]
y = housing_df.loc[:, "MedHouseVal"]



0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64


In [7]:
# Split using tts with 80/20 and 42 random state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

# Print shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)


In [8]:
# Create scaler object
scaler = StandardScaler()

print("Features before scaling: ")
print(X_train.head())

# Scale features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

print("Features after scaling: ")
print(X_train_scaled[:5])

Features before scaling: 
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
14196  3.2596      33.0  5.017657   1.006421      2300.0  3.691814     32.71   
8267   3.8125      49.0  4.473545   1.041005      1314.0  1.738095     33.77   
17445  4.1563       4.0  5.645833   0.985119       915.0  2.723214     34.66   
14265  1.9425      36.0  4.002817   1.033803      1418.0  3.994366     32.69   
2271   3.5542      43.0  6.268421   1.134211       874.0  2.300000     36.78   

       Longitude  
14196    -117.03  
8267     -118.16  
17445    -120.48  
14265    -117.11  
2271     -119.80  
Features after scaling: 
[[-0.326196    0.34849025 -0.17491646 -0.20836543  0.76827628  0.05137609
  -1.3728112   1.27258656]
 [-0.03584338  1.61811813 -0.40283542 -0.12853018 -0.09890135 -0.11736222
  -0.87669601  0.70916212]
 [ 0.14470145 -1.95271028  0.08821601 -0.25753771 -0.44981806 -0.03227969
  -0.46014647 -0.44760309]
 [-1.01786438  0.58654547 -0.60001532 -0.14515634 -